# 06 · Portfolio Transaction Simulation

Reads validated Parquet files produced by `scripts/normalize_portfolio_inputs.py`,
replays transactions through the FIFO lot engine, reconciles derived holdings
against a broker snapshot, and produces a per-security portfolio report.

**This notebook uses `src/portfolio_sim.py`.  
It does not import from `src/tax_risk_sim.py` or `src/inputs.py`.**

---

## Inputs

| File | Description |
|------|-------------|
| `data/examples/db_transactions_synthetic.csv` | 10-row synthetic transaction log |
| `data/examples/db_holdings_synthetic.csv` | Broker holdings snapshot (2023-12-31) |

Real data goes under `data/private/` (gitignored).  
Run `scripts/normalize_portfolio_inputs.py` first to produce validated Parquet.

> **FX note**: The ECB provider fetches live rates and requires internet access.  
> Substitute `make_fx_provider('yahoo')` for Yahoo Finance rates,  
> or use the `FixedRateFXProvider` stub defined in Cell 3 for offline demos.

In [ ]:
import subprocess
import sys
from pathlib import Path

# ── Paths ──────────────────────────────────────────────────────────────────────
WORK = Path('.').resolve()
if WORK.name == 'notebooks':
    WORK = WORK.parent   # run from repo root or notebooks/

SRC = WORK / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

TX_CSV      = WORK / 'data' / 'examples' / 'db_transactions_synthetic.csv'
HLD_CSV     = WORK / 'data' / 'examples' / 'db_holdings_synthetic.csv'
TX_PARQUET  = WORK / 'data' / 'examples' / 'db_transactions_synthetic.parquet'
HLD_PARQUET = WORK / 'data' / 'examples' / 'db_holdings_synthetic.parquet'

print(f'Repo root : {WORK}')
print(f'Tx CSV    : {TX_CSV.exists()}')
print(f'Hld CSV   : {HLD_CSV.exists()}')

In [ ]:
# ── Step 1: normalize CSVs → Parquet ──────────────────────────────────────────
# Skips files that already exist. Delete the .parquet files to force re-normalization.

NORMALIZE = WORK / 'scripts' / 'normalize_portfolio_inputs.py'

for csv_path, pq_path, schema_type in [
    (TX_CSV,  TX_PARQUET,  'transactions'),
    (HLD_CSV, HLD_PARQUET, 'holdings'),
]:
    if not pq_path.exists():
        result = subprocess.run(
            [
                sys.executable, str(NORMALIZE),
                '--input',          str(csv_path),
                '--output-csv',     str(csv_path.with_suffix('.normalized.csv')),
                '--output-parquet', str(pq_path),
                '--type',           schema_type,
            ],
            capture_output=True, text=True,
        )
        if result.returncode != 0:
            print(f'ERROR: {result.stderr}')
        else:
            print(result.stdout.strip())
    else:
        print(f'Already exists: {pq_path.name}')

In [ ]:
# ── Step 2: load validated Parquet ────────────────────────────────────────────

import pandas as pd

transactions = pd.read_parquet(TX_PARQUET)
holdings_snapshot = pd.read_parquet(HLD_PARQUET)

print(f'Transactions : {len(transactions)} rows')
display(transactions)

print(f'\nHoldings snapshot : {len(holdings_snapshot)} rows')
display(holdings_snapshot)

In [ ]:
# ── Step 3: FX provider ───────────────────────────────────────────────────────
#
# Default in this notebook: FixedRateFXProvider (offline, reproducible demo).
# For real data, swap in make_fx_provider('ecb') or make_fx_provider('yahoo').

from portfolio_sim import FXProvider, make_fx_provider

# -- Live ECB rates -----------------------------------------------------------
# fx = make_fx_provider('ecb')   # requires internet; 7-day lookback

# -- Live Yahoo Finance rates -------------------------------------------------
# fx = make_fx_provider('yahoo') # requires internet; 5-day lookback

# -- Fixed-rate stub (offline demo) -------------------------------------------
# Approximate EUR/USD = 0.92 as of end-2023. Replace with real rates for production.
class FixedRateFXProvider(FXProvider):
    """Simple fixed-rate stub for offline demos and testing."""
    _RATES: dict[tuple[str, str], float] = {
        ('USD', 'EUR'): 0.9200,
        ('EUR', 'USD'): 1.0870,
        ('GBP', 'EUR'): 1.1500,
        ('EUR', 'GBP'): 0.8696,
    }

    def rate(self, from_currency: str, to_currency: str, date: str) -> float:
        if from_currency == to_currency:
            return 1.0
        key = (from_currency, to_currency)
        if key in self._RATES:
            return self._RATES[key]
        # Cross via EUR
        to_eur = self.rate(from_currency, 'EUR', date)
        eur_to = self.rate('EUR', to_currency, date)
        return to_eur * eur_to

fx = FixedRateFXProvider()
print('FX provider: FixedRateFXProvider (offline demo)')
print(f'  USD → EUR : {fx.rate("USD", "EUR", "2023-12-31"):.4f}')
print(f'  EUR → USD : {fx.rate("EUR", "USD", "2023-12-31"):.4f}')

In [ ]:
# ── Step 4: configure simulation parameters ───────────────────────────────────
#
# current_prices_eur: per-security closing price in EUR on the reporting date.
# Source from the broker holdings snapshot or a market data provider.

REPORTING_DATE    = '2023-12-31'
CAPITAL_GAINS_TAX = 0.26375   # German Abgeltungsteuer 25% + Solidaritätszuschlag (simplified flat)
DIVIDEND_TAX      = 0.26375   # Same flat rate for dividend income in this simplified model

# Apple price is quoted in USD in the broker snapshot; convert to EUR for the simulation
apple_price_usd = 192.50   # from holdings_snapshot
apple_price_eur = fx.rate('USD', 'EUR', REPORTING_DATE) * apple_price_usd

CURRENT_PRICES_EUR: dict[str, float] = {
    'DE0005140008': 15.60,         # Deutsche Bank AG (EUR)
    'IE00B4L5Y983': 87.20,         # iShares Core MSCI World ETF (EUR)
    'US0378331005': apple_price_eur,  # Apple Inc (USD → EUR)
}

print(f'Reporting date : {REPORTING_DATE}')
print(f'Capital gains tax rate : {CAPITAL_GAINS_TAX:.5f}')
print(f'Dividend tax rate      : {DIVIDEND_TAX:.5f}')
print('\nCurrent prices (EUR):')
for isin, price in CURRENT_PRICES_EUR.items():
    print(f'  {isin}: {price:.4f}')

In [ ]:
# ── Step 5: check for unsupported corporate actions ───────────────────────────

from portfolio_sim import check_unsupported_actions

blocked_isins = check_unsupported_actions(transactions)

if blocked_isins:
    print(f'WARNING: {len(blocked_isins)} ISIN(s) blocked by unsupported corporate actions:')
    for isin in blocked_isins:
        print(f'  - {isin}')
    print('Partial results will exclude these securities.')
else:
    print('No unsupported corporate actions — full simulation available.')

In [ ]:
# ── Step 6: run portfolio simulation ─────────────────────────────────────────
#
# simulate_portfolio_partial is the safe default: excludes blocked ISINs instead
# of raising UnsupportedCorporateAction. With no blocked ISINs it is equivalent
# to simulate_portfolio.

from portfolio_sim import simulate_portfolio_partial

result, excluded = simulate_portfolio_partial(
    transactions=transactions,
    current_prices_eur=CURRENT_PRICES_EUR,
    capital_gains_tax_rate=CAPITAL_GAINS_TAX,
    dividend_tax_rate=DIVIDEND_TAX,
    fx_provider=fx,
    lot_method='fifo',
    reporting_date=REPORTING_DATE,
)

if excluded:
    print(f'PARTIAL RESULTS — excluded ISINs (unsupported actions): {excluded}')
    print('Totals below do NOT represent the full portfolio.\n')

display(result)

In [ ]:
# ── Step 7: portfolio totals ──────────────────────────────────────────────────

totals = result[[
    'market_value_eur',
    'unrealised_gain_eur',
    'realised_gain_ytd_eur',
    'tax_paid_ytd_eur',
]].sum()

print(f'Portfolio report — {REPORTING_DATE}')
print('=' * 45)
print(f'  Market value (EUR)      : {totals["market_value_eur"]:>12,.2f}')
print(f'  Unrealised gain (EUR)   : {totals["unrealised_gain_eur"]:>12,.2f}')
print(f'  Realised gain YTD (EUR) : {totals["realised_gain_ytd_eur"]:>12,.2f}')
print(f'  Tax paid YTD (EUR)      : {totals["tax_paid_ytd_eur"]:>12,.2f}')

if excluded:
    print()
    print(f'  PARTIAL — {len(excluded)} ISIN(s) excluded: {excluded}')

In [ ]:
# ── Step 8: holdings reconciliation ──────────────────────────────────────────
#
# Compare transaction-derived share counts against the broker snapshot.
# Status legend:
#   match        — derived vs broker difference <= 0.001 shares
#   mismatch     — difference > tolerance (investigate before trusting output)
#   derived_only — in transactions but not in broker snapshot
#   broker_only  — in broker snapshot but not in transactions

from portfolio_sim import (
    apply_buy,
    apply_sell_fifo,
    apply_split,
    derive_holdings_from_lots,
    reconcile_holdings,
)

# Replay transactions to rebuild the lot ledger for reconciliation
# (simulate_portfolio returns the output DataFrame, not the internal lots)
lots: list[dict] = []
for _, row in transactions.sort_values('date').iterrows():
    isin     = row['isin']
    tx_type  = row['transaction_type']
    currency = str(row['currency'])
    date     = str(row['date'])

    if tx_type == 'buy':
        price_eur = fx.convert(float(row['price']), currency, 'EUR', date)
        lots = apply_buy(lots, isin, date, price_eur, float(row['quantity']))
    elif tx_type == 'sell':
        price_eur = fx.convert(float(row['price']), currency, 'EUR', date)
        lots, _, _ = apply_sell_fifo(lots, isin, float(row['quantity']), price_eur, CAPITAL_GAINS_TAX)
    elif tx_type == 'split':
        lots = apply_split(lots, isin, float(row['quantity']))

lot_holdings    = derive_holdings_from_lots(lots)
broker_holdings = holdings_snapshot[['isin', 'quantity']].copy()

reconciliation = reconcile_holdings(lot_holdings, broker_holdings)
print('Holdings reconciliation:')
display(reconciliation)

mismatches = reconciliation[reconciliation['status'] == 'mismatch']
if mismatches.empty:
    print('All holdings match the broker snapshot (within tolerance).')
else:
    print(f'WARNING: {len(mismatches)} mismatch(es) — review before trusting simulation output.')
    display(mismatches)

In [ ]:
# ── Step 9: lot ledger with derived unrealised gain ───────────────────────────

from portfolio_sim import lots_to_dataframe

lot_df = lots_to_dataframe(lots)
lot_df['unrealised_gain_eur'] = lot_df.apply(
    lambda r: round(
        (CURRENT_PRICES_EUR.get(r['isin'], 0.0) - r['lot_price_eur']) * r['remaining_shares'], 2
    ),
    axis=1,
)

print('Open lot ledger with derived unrealised gain (computed at reporting date, not stored):')
display(lot_df)

## Known limitations (v1)

| Limitation | Note |
|------------|------|
| Flat tax rates | `CAPITAL_GAINS_TAX` and `DIVIDEND_TAX` applied uniformly. German Sparer-Pauschbetrag (€1,000/year exemption) is not modelled. |
| Solidarity surcharge | Not modelled separately — fold it into the flat rate if needed (26.375% = 25% × 1.055). |
| Reverse-split fractional shares | Retained at full precision; no cash distribution event generated. |
| Unsupported corporate actions | `merger`, `spin_off`, `option` block the affected ISIN from trusted totals. |
| Jurisdiction-aware tax | `jurisdiction` field captured but not yet used to vary tax logic by country. |
| Transaction-date FX | FX conversion uses the transaction date; intraday spot rates are not modelled. |

See `docs/portfolio-transaction-mvp/requirements-and-decisions.md` for full decisions and limitations.